# 10. Entrenamiento de Swin-T

Clasificación de 30 especies y `OTHER_SPECIES` a partir de espectrogramas 224 × 224. Se utiliza **Macro-F1** como métrica principal; `test` se mantiene reservado.

## 1. Entorno

Swin-T con pesos ImageNet y precisión mixta en GPU. El dataset es público y no requiere credenciales.

In [ ]:
%pip -q install kagglehub

import gc, itertools, json, os, random
from pathlib import Path
import kagglehub, matplotlib.pyplot as plt, numpy as np, pandas as pd, torch, torch.nn as nn, torchvision
from PIL import Image
from google.colab import drive
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import Swin_T_Weights, swin_t
from tqdm.auto import tqdm

SEED = 42
DATASET = "santiagocorrea802/clasificador-aves-espectrogramas-v1/versions/1"
BASE_LOCAL = Path("/content/clasificador_swin_t")
DIR_BASE = Path("/content/drive/MyDrive/Proyecto_Aves/experiments/classifier/C2_Swin_T")
NUM_CLASES = 31
BATCH_FISICO, ACUMULACION = 16, 2
WEIGHT_DECAY, LABEL_SMOOTHING = 0.01, 0.05
DEVICE = torch.device("cuda")
assert torch.cuda.is_available(), "Selecciona GPU T4 en Colab"

def fijar_semilla():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

fijar_semilla()
print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| Batch efectivo:", BATCH_FISICO * ACUMULACION)

## 2. Dataset y particiones

Se emplean todos los eventos elegibles de train y validation, separados por grabación. Los PNG ya están preprocesados; no se generan espectrogramas.

In [ ]:
DIR_DATOS = Path(kagglehub.dataset_download(DATASET, output_dir=str(BASE_LOCAL / "dataset")))
requeridos = ["manifest_train.csv", "manifest_validation.csv", "class_mapping.json", "preprocessing_config.json", "recording_split_seed0.csv"]
assert all((DIR_DATOS / f).is_file() for f in requeridos), "Dataset incompleto"
config_datos = json.loads((DIR_DATOS / "preprocessing_config.json").read_text(encoding="utf-8"))
assert config_datos["png_size"] == [224, 224] and config_datos["window_s"] == 8.0
assert config_datos["freq_crop_width_hz"] == 8000

train = pd.read_csv(DIR_DATOS / "manifest_train.csv")
val = pd.read_csv(DIR_DATOS / "manifest_validation.csv")
clases = json.loads((DIR_DATOS / "class_mapping.json").read_text(encoding="utf-8"))
codigos = {int(c["id"]): c["name"] for c in clases}
assert len(train) == 17352 and len(val) == 3631 and set(codigos) == set(range(NUM_CLASES))
assert codigos[0] == "OTHER_SPECIES"

def preparar(df, nombre):
    df = df.copy()
    assert df["split"].eq(nombre).all() and df["event_id"].is_unique and df["sample_id"].is_unique
    assert df["class_code"].isin(codigos).all() and df["class_label"].eq(df["class_code"].map(codigos)).all()
    elegible = df["eligible_primary"].astype(str).str.lower().map({"true": True, "false": False})
    assert elegible.notna().all(), "Valores invalidos en eligible_primary"
    return df.loc[elegible.astype(bool)].reset_index(drop=True)

train_uso, val_uso = preparar(train, "train"), preparar(val, "validation")
assert len(train_uso) == 17265 and len(val_uso) == 3608
assert set(train_uso.recording_uid).isdisjoint(set(val_uso.recording_uid))

def ruta_png(fila):
    return DIR_DATOS / Path(fila.archivo_tar).stem / fila.ruta_png

for nombre, df in (("train", train_uso), ("validation", val_uso)):
    faltantes = sum(not ruta_png(fila).is_file() for fila in df.itertuples(index=False))
    assert faltantes == 0, f"{nombre}: faltan {faltantes} imagenes"

print("Train:", len(train_uso), "| Validation:", len(val_uso), "| Clases:", len(codigos))
print("Separacion por grabacion y archivos PNG: correctos")

## 3. Entradas y funciones de entrenamiento

Se conserva la normalización ImageNet de EfficientNet-B0. Se realiza fine-tuning completo con AdamW y pérdida ponderada por clase.

In [ ]:
transformacion = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class DatasetAves(Dataset):
    def __init__(self, tabla):
        self.rutas = [ruta_png(f) for f in tabla.itertuples(index=False)]
        self.etiquetas = tabla["class_code"].astype(int).to_numpy()

    def __len__(self): return len(self.etiquetas)

    def __getitem__(self, i):
        with Image.open(self.rutas[i]) as imagen:
            x = transformacion(imagen.convert("RGB"))
        return x, int(self.etiquetas[i])

dataset_train, dataset_val = DatasetAves(train_uso), DatasetAves(val_uso)

def crear_loader_train(epoca):
    generador = torch.Generator().manual_seed(SEED + epoca)
    return DataLoader(dataset_train, batch_size=BATCH_FISICO, shuffle=True, num_workers=0, pin_memory=True, generator=generador)

loader_val = DataLoader(dataset_val, batch_size=BATCH_FISICO, shuffle=False, num_workers=0, pin_memory=True)

PESOS = Swin_T_Weights.IMAGENET1K_V1
conteos = np.bincount(train_uso["class_code"].to_numpy(), minlength=NUM_CLASES)
assert (conteos > 0).all()
pesos_clase = np.sqrt(len(train_uso) / (NUM_CLASES * conteos))
pesos_clase /= pesos_clase.mean()
criterio_train = nn.CrossEntropyLoss(weight=torch.tensor(pesos_clase, dtype=torch.float32, device=DEVICE), label_smoothing=LABEL_SMOOTHING)
criterio_val = nn.CrossEntropyLoss()

def crear_modelo(preentrenado=True):
    modelo = swin_t(weights=PESOS if preentrenado else None)
    modelo.head = nn.Linear(modelo.head.in_features, NUM_CLASES)
    return modelo.to(DEVICE)

def crear_optimizador(modelo, lr_backbone, lr_cabeza):
    grupos = [
        {"params": modelo.features.parameters(), "lr": lr_backbone},
        {"params": modelo.norm.parameters(), "lr": lr_backbone},
        {"params": modelo.head.parameters(), "lr": lr_cabeza}
    ]
    assert sum(p.numel() for g in grupos for p in g["params"]) == sum(p.numel() for p in modelo.parameters())
    return torch.optim.AdamW(grupos, weight_decay=WEIGHT_DECAY)

def guardar_checkpoint(objeto, destino):
    temporal = destino.with_suffix(".tmp")
    torch.save(objeto, temporal)
    os.replace(temporal, destino)

def entrenar_epoca(modelo, optimizador, scaler, epoca):
    modelo.train()
    iterador = iter(crear_loader_train(epoca))
    pasos = (len(iterador) + ACUMULACION - 1) // ACUMULACION
    perdida_total = 0.0
    for _ in tqdm(range(pasos), desc=f"Train {epoca}", leave=False):
        grupo = list(itertools.islice(iterador, ACUMULACION))
        n_grupo = sum(len(y) for _, y in grupo)
        optimizador.zero_grad(set_to_none=True)
        for x, y in grupo:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.float16):
                salida = modelo(x)
                perdida = criterio_train(salida, y)
            scaler.scale(perdida * (len(y) / n_grupo)).backward()
            perdida_total += perdida.item() * len(y)
        scaler.unscale_(optimizador)
        torch.nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)
        scaler.step(optimizador); scaler.update()
    return perdida_total / len(dataset_train)

@torch.inference_mode()
def evaluar(modelo):
    modelo.eval()
    suma, reales, predichos = 0.0, [], []
    for x, y in tqdm(loader_val, desc="Validation", leave=False):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            salida = modelo(x)
            perdida = criterio_val(salida, y)
        suma += perdida.item() * len(y)
        reales.extend(y.cpu().tolist()); predichos.extend(salida.argmax(1).cpu().tolist())
    return {
        "val_loss": suma / len(dataset_val),
        "macro_f1": float(f1_score(reales, predichos, labels=list(range(NUM_CLASES)), average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(reales, predichos))
    }

fijar_semilla()
modelo_prueba = crear_modelo()
print("Parametros Swin-T:", f"{sum(p.numel() for p in modelo_prueba.parameters()):,}")
print("Cabeza:", modelo_prueba.head)
del modelo_prueba
gc.collect(); torch.cuda.empty_cache()

## 4. Búsqueda de hiperparámetros

Cuatro combinaciones de tasas de aprendizaje, **6 épocas por configuración**, con idéntica inicialización y datos. Se selecciona el mayor Macro-F1 de validation.

In [ ]:
drive.mount("/content/drive")
DIR_HPO = DIR_BASE / "HPO_v1"
DIR_HPO.mkdir(parents=True, exist_ok=True)
EPOCHS_HPO = 6
BUSQUEDA = [
    {"nombre": "H1", "lr_backbone": 1e-5, "lr_cabeza": 1e-4},
    {"nombre": "H2", "lr_backbone": 2e-5, "lr_cabeza": 2e-4},
    {"nombre": "H3", "lr_backbone": 5e-5, "lr_cabeza": 5e-4},
    {"nombre": "H4", "lr_backbone": 2e-5, "lr_cabeza": 5e-4}
]
config_hpo = {
    "modelo": "Swin-T", "dataset": DATASET, "seed": SEED, "epochs": EPOCHS_HPO,
    "busqueda": BUSQUEDA, "batch_fisico": BATCH_FISICO, "acumulacion": ACUMULACION,
    "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
    "train_samples": len(train_uso), "validation_samples": len(val_uso)
}
ruta_config_hpo = DIR_HPO / "configuracion.json"
if ruta_config_hpo.exists():
    assert json.loads(ruta_config_hpo.read_text(encoding="utf-8")) == config_hpo, "HPO previo con otra configuracion"
else:
    ruta_config_hpo.write_text(json.dumps(config_hpo, indent=2), encoding="utf-8")

print("Configuraciones:", len(BUSQUEDA), "| Epocas por configuracion:", EPOCHS_HPO)
display(pd.DataFrame(BUSQUEDA))

In [ ]:
for config in BUSQUEDA:
    nombre = config["nombre"]
    ruta = DIR_HPO / f"{nombre}_metricas.csv"
    if ruta.exists() and len(pd.read_csv(ruta)) == EPOCHS_HPO:
        print(nombre, "ya completado")
        continue

    fijar_semilla()
    modelo = crear_modelo()
    optimizador = crear_optimizador(modelo, config["lr_backbone"], config["lr_cabeza"])
    scaler = torch.amp.GradScaler("cuda")
    registros = []
    print(f"\n{nombre}: backbone={config['lr_backbone']}, cabeza={config['lr_cabeza']}")

    for epoca in range(1, EPOCHS_HPO + 1):
        train_loss = entrenar_epoca(modelo, optimizador, scaler, epoca)
        metricas = evaluar(modelo)
        registros.append({"experimento": nombre, "epoca": epoca, "train_loss": train_loss, **metricas,
                          "lr_backbone": config["lr_backbone"], "lr_cabeza": config["lr_cabeza"]})
        temporal = ruta.with_suffix(".tmp")
        pd.DataFrame(registros).to_csv(temporal, index=False)
        os.replace(temporal, ruta)
        print(f"{nombre} {epoca:02d}/{EPOCHS_HPO} | Train={train_loss:.4f} | Val={metricas['val_loss']:.4f} | Macro-F1={metricas['macro_f1']:.4f} | Accuracy={metricas['accuracy']:.4f}")

    del modelo, optimizador, scaler
    gc.collect(); torch.cuda.empty_cache()

print("Busqueda de hiperparametros finalizada")

## 5. Comparación de configuraciones

Se guarda la tabla y las curvas de HPO. El entrenamiento definitivo comienza después desde ImageNet, no desde estos modelos.

In [ ]:
tablas = []
for config in BUSQUEDA:
    tabla = pd.read_csv(DIR_HPO / f"{config['nombre']}_metricas.csv")
    assert len(tabla) == EPOCHS_HPO, f"Experimento incompleto: {config['nombre']}"
    tablas.append(tabla)

historial_hpo = pd.concat(tablas, ignore_index=True)
resumen_hpo = (historial_hpo.sort_values(["macro_f1", "val_loss"], ascending=[False, True])
               .groupby("experimento", as_index=False).first().sort_values("macro_f1", ascending=False))
ganador = resumen_hpo.iloc[0]
seleccion = {"experimento": str(ganador["experimento"]), "epoca": int(ganador["epoca"]),
             "macro_f1": float(ganador["macro_f1"]), "lr_backbone": float(ganador["lr_backbone"]),
             "lr_cabeza": float(ganador["lr_cabeza"])}
resumen_hpo.to_csv(DIR_HPO / "resumen_hpo.csv", index=False)
historial_hpo.to_csv(DIR_HPO / "historial_hpo.csv", index=False)
(DIR_HPO / "seleccion.json").write_text(json.dumps(seleccion, indent=2), encoding="utf-8")

fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for nombre, grupo in historial_hpo.groupby("experimento"):
    for ax, columna, titulo in zip(ejes, ["macro_f1", "val_loss", "accuracy"], ["Macro-F1", "Validation loss", "Accuracy"]):
        ax.plot(grupo["epoca"], grupo[columna], marker="o", label=nombre)
        ax.set_title(titulo); ax.set_xlabel("Epoca"); ax.grid(alpha=0.3)
ejes[0].legend(); plt.tight_layout()
fig.savefig(DIR_HPO / "comparacion_hpo.png", dpi=160, bbox_inches="tight")
plt.show()
display(resumen_hpo[["experimento", "epoca", "macro_f1", "accuracy", "val_loss", "lr_backbone", "lr_cabeza"]].round(5))
print("Configuracion seleccionada:", seleccion)

## 6. Entrenamiento definitivo

Un único fine-tuning desde ImageNet, con **máximo 60 épocas** y **early stopping de 12**. `ReduceLROnPlateau` reduce el learning rate antes de detener el entrenamiento.

In [ ]:
DIR_FINAL = DIR_BASE / "final_v1"
DIR_FINAL.mkdir(parents=True, exist_ok=True)
RUTA_BEST = DIR_FINAL / "checkpoint_best.pth"
RUTA_LATEST = DIR_FINAL / "checkpoint_latest.pth"
RUTA_METRICAS = DIR_FINAL / "metricas.csv"
MAX_EPOCHS, PACIENCIA, MIN_DELTA = 60, 12, 0.001

seleccion = json.loads((DIR_HPO / "seleccion.json").read_text(encoding="utf-8"))
config_final = {
    "modelo": "Swin-T", "pesos": "IMAGENET1K_V1", "dataset": DATASET, "seed": SEED,
    "hpo_seleccionado": seleccion["experimento"],
    "lr_backbone": seleccion["lr_backbone"], "lr_cabeza": seleccion["lr_cabeza"],
    "batch_fisico": BATCH_FISICO, "acumulacion": ACUMULACION,
    "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
    "max_epochs": MAX_EPOCHS, "patience": PACIENCIA, "min_delta": MIN_DELTA,
    "train_samples": len(train_uso), "validation_samples": len(val_uso)
}
ruta_config_final = DIR_FINAL / "configuracion.json"
if ruta_config_final.exists():
    assert json.loads(ruta_config_final.read_text(encoding="utf-8")) == config_final, "Entrenamiento guardado con otra configuracion"
else:
    ruta_config_final.write_text(json.dumps(config_final, indent=2), encoding="utf-8")

print("HPO ganador:", seleccion["experimento"], "| Backbone:", seleccion["lr_backbone"], "| Cabeza:", seleccion["lr_cabeza"])
print("Maximo de epocas:", MAX_EPOCHS, "| Paciencia:", PACIENCIA, "| Mejora minima:", MIN_DELTA)

In [ ]:
fijar_semilla()
modelo = crear_modelo(preentrenado=not RUTA_LATEST.exists())
optimizador = crear_optimizador(modelo, config_final["lr_backbone"], config_final["lr_cabeza"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizador, mode="max", factor=0.5, patience=3, threshold=MIN_DELTA, threshold_mode="abs", min_lr=1e-6
)
scaler = torch.amp.GradScaler("cuda")
historial, mejor_f1, sin_mejora, inicio, completado = [], -1.0, 0, 1, False

if RUTA_LATEST.exists():
    estado = torch.load(RUTA_LATEST, map_location="cpu", weights_only=True)
    assert estado["config"] == config_final, "Checkpoint incompatible"
    modelo.load_state_dict(estado["modelo"])
    optimizador.load_state_dict(estado["optimizador"])
    scheduler.load_state_dict(estado["scheduler"])
    scaler.load_state_dict(estado["scaler"])
    torch.set_rng_state(estado["rng_cpu"].cpu())
    torch.cuda.set_rng_state_all([s.cpu() for s in estado["rng_cuda"]])
    historial, mejor_f1 = estado["historial"], float(estado["mejor_f1"])
    sin_mejora, completado = int(estado["sin_mejora"]), bool(estado["completado"])
    inicio = int(estado["epoca"]) + 1
    print("Reanudando en epoca:", inicio)
else:
    print("Iniciando entrenamiento definitivo desde ImageNet")

if not completado:
    for epoca in range(inicio, MAX_EPOCHS + 1):
        train_loss = entrenar_epoca(modelo, optimizador, scaler, epoca)
        metricas = evaluar(modelo)
        historial.append({"epoca": epoca, "train_loss": train_loss, **metricas,
                          "lr_backbone": optimizador.param_groups[0]["lr"], "lr_cabeza": optimizador.param_groups[2]["lr"]})
        scheduler.step(metricas["macro_f1"])

        if metricas["macro_f1"] > mejor_f1 + MIN_DELTA:
            mejor_f1, sin_mejora = metricas["macro_f1"], 0
            guardar_checkpoint({"model_state_dict": modelo.state_dict(), "epoca": epoca,
                                 "macro_f1": mejor_f1, "config": config_final}, RUTA_BEST)
            print("Nuevo mejor checkpoint")
        else:
            sin_mejora += 1

        completado = (sin_mejora >= PACIENCIA or epoca >= MAX_EPOCHS)
        temporal_csv = RUTA_METRICAS.with_suffix(".tmp")
        pd.DataFrame(historial).to_csv(temporal_csv, index=False)
        os.replace(temporal_csv, RUTA_METRICAS)
        guardar_checkpoint({
            "modelo": modelo.state_dict(), "optimizador": optimizador.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all(),
            "epoca": epoca, "mejor_f1": mejor_f1, "sin_mejora": sin_mejora,
            "historial": historial, "config": config_final, "completado": completado
        }, RUTA_LATEST)

        print(f"Epoca {epoca:02d}/{MAX_EPOCHS} | Train={train_loss:.4f} | Val={metricas['val_loss']:.4f} | Macro-F1={metricas['macro_f1']:.4f} | Accuracy={metricas['accuracy']:.4f} | Sin mejora={sin_mejora}/{PACIENCIA}")
        if completado:
            print("Entrenamiento finalizado:", "early stopping" if sin_mejora >= PACIENCIA else "maximo de epocas")
            break
else:
    print("El entrenamiento ya habia finalizado")

assert RUTA_BEST.exists(), "Falta checkpoint del mejor modelo"
print("Mejor Macro-F1:", round(mejor_f1, 4), "| Checkpoint:", RUTA_BEST)

## 7. Curvas de aprendizaje

Se conservan las métricas de cada época y el checkpoint con mayor Macro-F1.

In [ ]:
metricas_final = pd.read_csv(RUTA_METRICAS)
fig, ejes = plt.subplots(2, 2, figsize=(13, 8))
for ax, columna, titulo in zip(ejes.flat[:3], ["macro_f1", "val_loss", "accuracy"], ["Macro-F1", "Validation loss", "Accuracy"]):
    ax.plot(metricas_final["epoca"], metricas_final[columna], marker=".")
    ax.set_title(titulo); ax.set_xlabel("Epoca"); ax.grid(alpha=0.3)
ejes.flat[3].plot(metricas_final["epoca"], metricas_final["lr_backbone"], label="Backbone")
ejes.flat[3].plot(metricas_final["epoca"], metricas_final["lr_cabeza"], label="Cabeza")
ejes.flat[3].set_yscale("log"); ejes.flat[3].set_title("Learning rates")
ejes.flat[3].set_xlabel("Epoca"); ejes.flat[3].legend(); ejes.flat[3].grid(alpha=0.3)
plt.tight_layout()
fig.savefig(DIR_FINAL / "curvas_entrenamiento.png", dpi=160, bbox_inches="tight")
plt.show()
print("Mejor epoca:", torch.load(RUTA_BEST, map_location="cpu", weights_only=True)["epoca"])

## 8. Evaluación por especie y fuente

Se evalúa el mejor checkpoint únicamente en validation: precisión, recall, F1 por clase, matriz de confusión y análisis por corpus.

In [ ]:
del modelo, optimizador, scheduler, scaler
gc.collect(); torch.cuda.empty_cache()

checkpoint = torch.load(RUTA_BEST, map_location="cpu", weights_only=True)
modelo_eval = crear_modelo(preentrenado=False)
modelo_eval.load_state_dict(checkpoint["model_state_dict"])
modelo_eval.eval()
y_true, y_pred, confianzas = [], [], []

with torch.inference_mode():
    for x, y in tqdm(loader_val, desc="Evaluacion final"):
        with torch.autocast("cuda", dtype=torch.float16):
            salida = modelo_eval(x.to(DEVICE, non_blocking=True))
        prob = salida.softmax(dim=1)
        confianza, clase = prob.max(dim=1)
        y_true.extend(y.tolist()); y_pred.extend(clase.cpu().tolist()); confianzas.extend(confianza.cpu().tolist())

resultados = val_uso.copy()
assert len(resultados) == len(y_true) == len(y_pred)
assert np.array_equal(resultados["class_code"].to_numpy(), np.asarray(y_true))
resultados["pred_code"], resultados["confianza"] = y_pred, confianzas
resultados["pred_label"] = resultados["pred_code"].map(codigos)
resultados["correcto"] = resultados["class_code"].eq(resultados["pred_code"])
resultados.to_csv(DIR_FINAL / "predicciones_validation.csv", index=False)

reporte = classification_report(y_true, y_pred, labels=list(range(NUM_CLASES)), output_dict=True, zero_division=0)
metricas_clase = pd.DataFrame([{
    "codigo": i, "especie": codigos[i], "n_train": int(conteos[i]),
    "n_validation": int(reporte[str(i)]["support"]), "precision": reporte[str(i)]["precision"],
    "recall": reporte[str(i)]["recall"], "f1": reporte[str(i)]["f1-score"]
} for i in range(NUM_CLASES)]).sort_values("f1")
metricas_clase.to_csv(DIR_FINAL / "metricas_por_clase.csv", index=False)
print("Checkpoint epoca:", checkpoint["epoca"], "| Macro-F1:", round(reporte["macro avg"]["f1-score"], 4))
print("Macro-Precision:", round(reporte["macro avg"]["precision"], 4), "| Macro-Recall:", round(reporte["macro avg"]["recall"], 4))
print("Weighted-F1:", round(reporte["weighted avg"]["f1-score"], 4), "| Accuracy:", round(accuracy_score(y_true, y_pred), 4))
display(metricas_clase.round(4))
assert abs(reporte["macro avg"]["f1-score"] - checkpoint["macro_f1"]) < 0.002, "Metricas y checkpoint difieren"

In [ ]:
matriz = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASES)))
matriz_norm = matriz / np.maximum(matriz.sum(axis=1, keepdims=True), 1)
pd.DataFrame(matriz, index=[codigos[i] for i in range(NUM_CLASES)], columns=[codigos[i] for i in range(NUM_CLASES)]).to_csv(DIR_FINAL / "matriz_confusion.csv")

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(matriz_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASES)); ax.set_yticks(range(NUM_CLASES))
ax.set_xticklabels(range(NUM_CLASES), fontsize=8); ax.set_yticklabels(range(NUM_CLASES), fontsize=8)
ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real")
ax.set_title("Swin-T - matriz de confusion normalizada")
fig.colorbar(im, ax=ax, label="Proporcion"); plt.tight_layout()
fig.savefig(DIR_FINAL / "matriz_confusion.png", dpi=160, bbox_inches="tight")
plt.show()

confusiones = pd.DataFrame([{"clase_real": codigos[i], "clase_predicha": codigos[j], "eventos": int(matriz[i, j])
                            } for i in range(NUM_CLASES) for j in range(NUM_CLASES) if i != j and matriz[i, j] > 0])
confusiones = confusiones.sort_values("eventos", ascending=False)
confusiones.to_csv(DIR_FINAL / "confusiones_frecuentes.csv", index=False)
display(confusiones.head(15))

col_fuente = next((c for c in ("dataset", "dataset_origen", "source_dataset", "corpus") if c in resultados.columns), None)
if col_fuente:
    filas = []
    for fuente, grupo in resultados.groupby(col_fuente):
        clases_presentes = sorted(grupo["class_code"].unique())
        filas.append({"fuente": fuente, "eventos": len(grupo), "clases_presentes": len(clases_presentes),
                      "accuracy": accuracy_score(grupo["class_code"], grupo["pred_code"]),
                      "macro_f1_clases_presentes": f1_score(grupo["class_code"], grupo["pred_code"], labels=clases_presentes, average="macro", zero_division=0)})
    tabla_fuentes = pd.DataFrame(filas)
    tabla_fuentes.to_csv(DIR_FINAL / "metricas_por_fuente.csv", index=False)
    display(tabla_fuentes.round(4))
    print("Los corpus tienen distribuciones de clases diferentes; sus F1 no son directamente comparables.")
else:
    print("No se encontro columna de corpus en el manifest")

## 9. Cierre

Se guardan los resultados y se libera la GPU. `test` permanecerá reservado hasta completar la comparación de arquitecturas.

In [ ]:
from google.colab import runtime

resumen = {
    "modelo": "Swin-T", "checkpoint_epoca": int(checkpoint["epoca"]),
    "macro_f1": float(reporte["macro avg"]["f1-score"]),
    "macro_precision": float(reporte["macro avg"]["precision"]),
    "macro_recall": float(reporte["macro avg"]["recall"]),
    "weighted_f1": float(reporte["weighted avg"]["f1-score"]),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "eventos_validation": len(y_true), "test_utilizado": False
}
(DIR_FINAL / "resumen_evaluacion.json").write_text(json.dumps(resumen, indent=2), encoding="utf-8")
print("Resultados guardados:", DIR_FINAL)
print("Validation Macro-F1:", round(resumen["macro_f1"], 4))
drive.flush_and_unmount()
runtime.unassign()